In [3]:
import re, json
import pandas as pd
from pathlib import Path

# ========= EDIT THESE =========
SRC = Path("./data_labels.csv")   # your input file
OUT = Path("./labels_train.csv")  # output template for Step 2
PATH_COL = "Data Path"  # <-- SET THIS to the correct column in your CSV
# If your noisy/target arrays are already in a single JSON column, set these:
NOISY_JSON_COL  = None   # e.g. "noisy_z_json" or "noisy"
TARGET_JSON_COL = None   # e.g. "target_y_json" or "ideal"
# If you want to force exact wide column lists, put them here (optional):
EXPLICIT_NOISY_COLS  = ['Noisy']  # e.g. ["noisy_q0","noisy_q1","noisy_q2","noisy_q3","noisy_q4"]
EXPLICIT_TARGET_COLS = ['Ideal']  # e.g. ["ideal_q0","ideal_q1","ideal_q2","ideal_q3","ideal_q4"]'Ideal', 'Noisy', 'Data Path'
# ==========================

df = pd.read_csv(SRC)
if PATH_COL not in df.columns:
    raise ValueError(f"PATH_COL='{PATH_COL}' not found. Available: {list(df.columns)}")
paths = df[PATH_COL].astype(str).str.strip()

def looks_like_json_array(val: str) -> bool:
    if pd.isna(val): return False
    s = str(val).strip()
    return s.startswith("[") and s.endswith("]")

def clean_array_str(s: str) -> str:
    s = str(s).strip()
    s = s.replace(";", ",")
    s = re.sub(r"(\d)\s+(-?\d)", r"\1,\2", s)   # "[0.1 0.2]" -> "[0.1,0.2]"
    s = re.sub(r",\s*,+", ",", s)
    return s

def parse_array_cell(x):
    if pd.isna(x): return []
    s = clean_array_str(x)
    if s.startswith("[") and s.endswith("]"):
        try:
            arr = json.loads(s)
            if not isinstance(arr, list):
                arr = [arr]
            return [float(v) for v in arr]
        except Exception:
            pass
    # last resort: split by comma/space
    parts = [p for p in re.split(r"[,\s]+", s.strip("[] ")) if p]
    out = []
    for p in parts:
        try: out.append(float(p))
        except Exception: out.append(None)
    return out

def detect_wide(df, prefixes):
    hits = []
    for c in df.columns:
        lc = c.lower().strip()
        for p in prefixes:
            p = p.lower()
            m = re.match(rf"^{re.escape(p)}(?:[_\-]?z)?(?:[_\-]?q)?(\d+)$", lc)  # noisy_q0, noisyzq3, etc.
            if m:
                hits.append((int(m.group(1)), c))
                break
    hits.sort(key=lambda x: x[0])
    return [c for _, c in hits]

def pick_json_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            # verify it looks like arrays
            sample = df[c].dropna().astype(str).head(5).tolist()
            if sample and any(looks_like_json_array(v) for v in sample):
                return c
    return None

def get_noisy_lists(df):
    # 1) explicit overrides
    if EXPLICIT_NOY_COLS := EXPLICIT_NOISY_COLS:
        cols = EXPLICIT_NOY_COLS
        vals = []
        for _, row in df.iterrows():
            vals.append([None if pd.isna(row[c]) or str(row[c]).strip()=="" else float(row[c]) for c in cols])
        return vals, f"wide (explicit): {cols}"
    if EXPLICIT_NOISY_JSON_COL and EXPLICIT_NOISY_JSON_COL in df.columns:
        return df[EXPLICIT_NOISY_JSON_COL].apply(parse_array_cell).tolist(), f"json: {EXPLICIT_NOISY_JSON_COL}"

    # 2) try json-ish single columns (now includes 'noisy_z')
    c = pick_json_col(df, ["noisy_z_json","noisy_json","noisy_z","noisy"])
    if c:
        return df[c].apply(parse_array_cell).tolist(), f"json: {c}"

    # 3) try wide (multiple scalar cols)
    cols = detect_wide(df, ["noisy","noisy_z","y_noisy"])
    if cols:
        # If there's exactly ONE column and it contains JSON arrays, treat it as JSON, not wide:
        if len(cols) == 1 and df[cols[0]].astype(str).apply(looks_like_json_array).any():
            c = cols[0]
            return df[c].apply(parse_array_cell).tolist(), f"json-like-in-wide: {c}"
        vals = []
        for _, row in df.iterrows():
            vals.append([None if pd.isna(row[c]) or str(row[c]).strip()=="" else float(row[c]) for c in cols])
        return vals, f"wide (detected): {cols}"

    raise ValueError("Could not detect NOISY columns. Set EXPLICIT_NOISY_* or verify column names.")

def get_target_lists(df):
    if EXPLICIT_TGT_COLS := EXPLICIT_TARGET_COLS:
        cols = EXPLICIT_TGT_COLS
        vals = []
        for _, row in df.iterrows():
            vals.append([None if pd.isna(row[c]) or str(row[c]).strip()=="" else float(row[c]) for c in cols])
        return vals, f"wide (explicit): {cols}"
    if EXPLICIT_TARGET_JSON_COL and EXPLICIT_TARGET_JSON_COL in df.columns:
        return df[EXPLICIT_TARGET_JSON_COL].apply(parse_array_cell).tolist(), f"json: {EXPLICIT_TARGET_JSON_COL}"

    c = pick_json_col(df, ["target_y_json","target_json","ideal_json","ideal","target","y_true","y"])
    if c:
        return df[c].apply(parse_array_cell).tolist(), f"json: {c}"

    cols = detect_wide(df, ["target","ideal","clean","y_true","label","y"])
    if cols:
        if len(cols) == 1 and df[cols[0]].astype(str).apply(looks_like_json_array).any():
            c = cols[0]
            return df[c].apply(parse_array_cell).tolist(), f"json-like-in-wide: {c}"
        vals = []
        for _, row in df.iterrows():
            vals.append([None if pd.isna(row[c]) or str(row[c]).strip()=="" else float(row[c]) for c in cols])
        return vals, f"wide (detected): {cols}"

    raise ValueError("Could not detect TARGET/IDEAL columns. Set EXPLICIT_TARGET_* or verify column names.")

# Build arrays:
noisy_lists, noisy_src = get_noisy_lists(df)
target_lists, target_src = get_target_lists(df)
print(f"Detected noisy from:  {noisy_src}")
print(f"Detected target from: {target_src}")

# Sanity: lengths and non-empty
if len(noisy_lists) != len(target_lists) or len(noisy_lists) != len(paths):
    raise ValueError("Row count mismatch among path/noisy/target.")

def infer_M(arrs, name):
    for a in arrs:
        if a and any(v is not None for v in a):
            return len(a)
    raise ValueError(f"All {name} arrays empty; detection wrong.")

Mn = infer_M(noisy_lists, "noisy")
Mt = infer_M(target_lists, "target")
if Mn != Mt:
    raise ValueError(f"M mismatch: noisy={Mn} vs target={Mt}")
M = Mn

def finalize(arrs, name):
    # pad/trim to M, and ensure at least one numeric value exists overall
    any_numeric = False
    out = []
    for a in arrs:
        b = list(a)
        if len(b) < M: b += [None]*(M-len(b))
        if len(b) > M: b = b[:M]
        bb = []
        for v in b:
            if v is None or (isinstance(v, float) and pd.isna(v)):
                bb.append(0.0)  # fill missing *within* an otherwise valid row
            else:
                any_numeric = True
                bb.append(float(v))
        out.append(bb)
    if not any_numeric:
        raise ValueError(f"No numeric values found in {name} columns; won’t write all-zero arrays.")
    return out

noisy_final  = finalize(noisy_lists,  "noisy")
target_final = finalize(target_lists, "target")

out_df = pd.DataFrame({
    "circuit_path": paths,
    "noisy_z_json": [json.dumps(row) for row in noisy_final],
    "target_y_json": [json.dumps(row) for row in target_final],
})
out_df.to_csv(OUT, index=False)
print(f"Wrote {OUT} with {len(out_df)} rows and M={M}.")
print(out_df.head(3).to_string(index=False))


ValueError: could not convert string to float: '[0.02324, -0.019200000000000002, -0.025840000000000002, 0.03704, 0.02832]'